# Recommendation System

### Data Description:

In [1]:
## ● anime_id = Unique ID of each anime.
## ● name = Anime title.
## ● type = Anime broadcast type, such as TV, OVA, etc.
## ● genre = Anime genre.
## ● episodes = The number of episodes of each anime.
## ● rating = The average rating for each anime compared to the number of users who gave ratings.
## ● members = Number of community members for each anime.

### Objective:

In [2]:
## ● The objective of this assignment is to implement a recommendation system using cosine similarity on an anime dataset.
## ● Use the Anime Dataset which contains information about various anime, including their titles, genres, No. of episodes and user ratings etc.

## Tasks:

### Data Preprocessing:

In [3]:
## ● Load the dataset into a suitable data structure (e.g., pandas DataFrame).
## ● Handle missing values, if any.
## ● Explore the dataset to understand its structure and attributes.

In [4]:
## Our core idea in this assignment/project is to recommend similar anime, based on features using Cosine Similarity 

In [5]:
## Import required libraries..
import pandas as pd
import numpy as np

In [6]:
## Load and display the dataset..
df = pd.read_csv("anime.csv")
df.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  object 
 4   episodes  12294 non-null  object 
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 672.5+ KB


In [8]:
df.isnull().sum()

## Some missing values can be observed in the data..
## genre-->62 missing values, type-->25 missing values, ratings-->230 missing values..

anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64

In [9]:
df.describe()

,anime_id,rating,members
count,12294.000000,12064.000000,1.229400e+04
mean,14058.221653,6.473902,1.807134e+04
std,11455.294701,1.026746,5.482068e+04
min,1.000000,1.670000,5.000000e+00
25%,3484.250000,5.880000,2.250000e+02
50%,10260.500000,6.570000,1.550000e+03
75%,24794.500000,7.180000,9.437000e+03
max,34527.000000,10.000000,1.013917e+06


In [10]:
## Handling of missing values..
## For genre, we have to drop those records with NA because it is a core feature, we can't fill it randomly..
## Since there are 12000+ records availble so we safely drop them.. 

df = df.dropna(subset = ['genre'])
df.isnull().sum()

anime_id      0
name          0
genre         0
type         22
episodes      0
rating      215
members       0
dtype: int64

In [11]:
## Shape of the dataset..
df.shape

(12232, 7)

In [12]:
df['type'].value_counts()

type
TV         3777
OVA        3310
Movie      2306
Special    1674
ONA         655
Music       488
Name: count, dtype: int64

In [13]:
## Since after the removal of null values from genre, only 22 null values are left in type..
## type is also kind of a category where it wouldn't be a good idea to fill values randomly..
## and ratio of null values to total number of remaining records is also very small.. 22/12232 = 0.001798..
## so we can safely remove those records as well which have 'type' value as NA without loosing much information..
df = df.dropna(subset = ['type'])
df.isnull().sum()

anime_id      0
name          0
genre         0
type          0
episodes      0
rating      193
members       0
dtype: int64

In [14]:
df.shape

(12210, 7)

In [15]:
## Now only 192 NA values are present in 'rating' column after removal of null values from 'type'
## this number is also very small considering the total number of records left (12210)..
## But let's not loose any more information since we can easily replace these NA values in this column using mean value..

df['rating'].fillna(df['rating'].mean(), inplace = True)
print(df.isnull().sum())
print(df.shape)

anime_id    0
name        0
genre       0
type        0
episodes    0
rating      0
members     0
dtype: int64
(12210, 7)


C:\Users\devlok\AppData\Local\Temp\ipykernel_8288\1636895834.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['rating'].fillna(df['rating'].mean(), inplace = True)


In [16]:
## if above code is not working then try the same using-->
## df.fillna({df['rating']: df['rating'].mean()}, inplace=True) 
## or
## df['rating'] = df['rating'].fillna(df['ratings'].mean())

## After this we succesfully handeled all the null values present in the dataset..

In [17]:
## now lets check for duplicates.. 
print("Shape of the dataset before dropping Duplicates:", df.shape)
df.drop_duplicates(inplace = True)
print("Shape of the dataset after dropping Duplicates:", df.shape)

## Since the shape remained same means no duplicate were present in the dataset..

Shape of the dataset before dropping Duplicates: (12210, 7)
Shape of the dataset after dropping Duplicates: (12210, 7)


In [18]:
## another way to check for duplicates, since python is case sensitive.. it will consider 'pokemon' and 'Pokemon' differently.. lets fix this..
## also try to remove any unnecessary spaces if present..
df['genre'] = df['genre'].str.lower().str.strip()

In [19]:
## again check for duplicates one last time..
df.drop_duplicates(inplace = True)
df.shape

(12210, 7)

In [20]:
## hence it is very cleared that there are indeed no duplicates present in the data..
## now final check..
print(df.isnull().sum())

anime_id    0
name        0
genre       0
type        0
episodes    0
rating      0
members     0
dtype: int64


### Feature Extraction:

In [21]:
## ● Decide on the features that will be used for computing similarity (e.g., genres, user ratings).
## ● Convert categorical features into numerical representations if necessary.
## ● Normalize numerical features if required

In [22]:
## This step will decide how good our recommendation system will be.. 
## Now we have to do some important conversions of features..
## genre.. text-->numeric

In [23]:
## lets convert genre to list.. 
df['genre'] = df['genre'].apply(lambda x: x.split(','))
df['genre']

0                [drama,  romance,  school,  supernatural]
1        [action,  adventure,  drama,  fantasy,  magic,...
2        [action,  comedy,  historical,  parody,  samur...
3                                      [sci-fi,  thriller]
4        [action,  comedy,  historical,  parody,  samur...
                               ...                        
12289                                             [hentai]
12290                                             [hentai]
12291                                             [hentai]
12292                                             [hentai]
12293                                             [hentai]
Name: genre, Length: 12210, dtype: object

In [24]:
## apply multilablebinarizer..
from sklearn.preprocessing import MultiLabelBinarizer
mlb = MultiLabelBinarizer()
genre_encoded = mlb.fit_transform(df['genre'])

In [25]:
## store genre_encoded as dataframe..
genre_df = pd.DataFrame(genre_encoded, columns = mlb.classes_)

In [26]:
genre_df.head()

,adventure,cars,comedy,dementia,demons,drama,ecchi,fantasy,game,harem,...,shoujo,shounen,slice of life,space,sports,super power,supernatural,thriller,vampire,yaoi
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [27]:
## Now we have succesfully encoded the genre column in binary numerical form, each column for each type..

In [28]:
## 'episodes' is in object form.. so we have to convert it into numeric before scaling it..
df['episodes'] = pd.to_numeric(df['episodes'], errors = 'coerce')

In [29]:
## Above step will convert numbers correctly.. but in case of unknown it will convert it to NaN...
## Lets check if any NaN is present or not.. 
df['episodes'].isnull().sum()

np.int64(307)

In [30]:
## Since there are 307 null values present in episodes after conversion to numeric datatype..
## Lets replace them with median instead of dropping it.. because i dont want to loose anymore informations.. 
## Also i have already prepared genre_df.. so removing records according to null values in episode will distrub the process unneccessarly..
## I will have to again rebuild genre_df if i remove records with episodes == NaN..

df['episodes'].fillna(df['episodes'].median(), inplace = True)
## or
## df['episodes'] = df['episodes'].fillna(df['episodes'].median())

C:\Users\devlok\AppData\Local\Temp\ipykernel_8288\3832862767.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['episodes'].fillna(df['episodes'].median(), inplace = True)


In [31]:
## Now next thing we have to do is bring ratings and episodes in similar scale..
## We will use minmax scaler..normalization..
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
num_features = scaler.fit_transform(df[['rating', 'episodes', 'members']])

In [32]:
## Store num_features also to a dataframe
num_df = pd.DataFrame(num_features, columns = ['rating', 'episodes', 'members'])
num_df.head()

,rating,episodes,members
0,0.924370,0.000000,0.197872
1,0.911164,0.034673,0.782770
2,0.909964,0.027518,0.112689
3,0.900360,0.012658,0.664325
4,0.899160,0.027518,0.149186


In [33]:
## Values are successfull scaled now.. 
## Now combine all features to one dataframe..
feature_matrix = pd.concat([genre_df, num_df], axis = 1)
feature_matrix.head()

,adventure,cars,comedy,dementia,demons,drama,ecchi,fantasy,game,harem,...,space,sports,super power,supernatural,thriller,vampire,yaoi,rating,episodes,members
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0.924370,0.000000,0.197872
1,1,0,0,0,0,1,0,1,0,0,...,0,0,0,0,0,0,0,0.911164,0.034673,0.782770
2,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0.909964,0.027518,0.112689
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0.900360,0.012658,0.664325
4,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0.899160,0.027518,0.149186


In [34]:
feature_matrix.shape

(12210, 85)

### Recommendation System:

In [35]:
## ● Design a function to recommend anime based on cosine similarity.
## ● Given a target anime, recommend a list of similar anime based on cosine similarity scores.
## ● Experiment with different threshold values for similarity scores to adjust the recommendation list size.

In [36]:
## lets compute similarity.. cosine similarity..
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(feature_matrix)

In [37]:
similarity_matrix.shape

(12210, 12210)

In [38]:
## The similarity_matrix that i created above is a matrix that compares each anime with every other anime..
## It means each anime can now be compared with all others..
## Value ranges from 0 to 1.. 0 means completely different and 1 means identical..

In [39]:
## Now lets build a function that recommend anime based on cosine similarity..
def recommend_anime(anime_name, df, similarity_matrix):
    index = df[df['name'] == anime_name].index[0]
    similarity_score = list(enumerate(similarity_matrix[index]))
    similarity_score = sorted(similarity_score, key = lambda x: x[1], reverse = True)
    similarity_score = similarity_score[1:11]
    anime_index = [i[0] for i in similarity_score]
    return df['name'].iloc[anime_index]

In [40]:
## I tried an anime name and got an error saying (index 12289 (from df) is out bounds for axis 0 with the size 12210)..
## It probably because df index is not reset.. even though df.shape = (12210, 7), index looks like (0, 1, 2, 3......12289, 12290, 12291, 12292, 12293)...
## But at the same time similarity matrix uses (0-->12209) continious without skipping..
## So lets reset it..

df = df.reset_index(drop = True)

## This aligns df index with similarity matrix..

In [41]:
df.shape

(12210, 7)

In [42]:
df

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"[drama, romance, school, supernatural]",Movie,1.0,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"[action, adventure, drama, fantasy, magic,...",TV,64.0,9.26,793665
2,28977,Gintama°,"[action, comedy, historical, parody, samur...",TV,51.0,9.25,114262
3,9253,Steins;Gate,"[sci-fi, thriller]",TV,24.0,9.17,673572
4,9969,Gintama&#039;,"[action, comedy, historical, parody, samur...",TV,51.0,9.16,151266
...,...,...,...,...,...,...,...
12205,9316,Toushindai My Lover: Minami tai Mecha-Minami,[hentai],OVA,1.0,4.15,211
12206,5543,Under World,[hentai],OVA,1.0,4.28,183
12207,5621,Violence Gekiga David no Hoshi,[hentai],OVA,4.0,4.88,219
12208,6133,Violence Gekiga Shin David no Hoshi: Inma Dens...,[hentai],OVA,1.0,4.98,175


In [43]:
## Test 
recommend_anime("Gintama°", df, similarity_matrix)

4                                            Gintama&#039;
9                                 Gintama&#039;: Enchousen
8        Gintama Movie: Kanketsu-hen - Yorozuya yo Eien...
65                  Gintama Movie: Shinyaku Benizakura-hen
63             Gintama: Yorinuki Gintama-san on Theater 2D
216                       Gintama: Shinyaku Benizakura-hen
306                       Gintama: Jump Festa 2014 Special
12                                                 Gintama
10849                                       Gintama (2017)
380      Gintama: Nanigoto mo Saiyo ga Kanjin nano de T...
Name: name, dtype: object

In [44]:
## I tried an anime name and got an error saying (index 12289 (from df) is out bounds for axis 0 with the size 12210)..
## It probably because df index is not reset.. even though df.shape = (12210, 7), index looks like (0, 1, 2, 3......12289, 12290, 12291, 12292, 12293)...
## But at the same time similarity matrix uses (0-->12209) continious without skipping..
## So lets reset it..
df = df.reset_index(drop = True)
## This aligns df index with similarity matrix..

In [45]:
## Test
recommend_anime("Gintama°", df, similarity_matrix)

4                                            Gintama&#039;
9                                 Gintama&#039;: Enchousen
8        Gintama Movie: Kanketsu-hen - Yorozuya yo Eien...
65                  Gintama Movie: Shinyaku Benizakura-hen
63             Gintama: Yorinuki Gintama-san on Theater 2D
216                       Gintama: Shinyaku Benizakura-hen
306                       Gintama: Jump Festa 2014 Special
12                                                 Gintama
10849                                       Gintama (2017)
380      Gintama: Nanigoto mo Saiyo ga Kanjin nano de T...
Name: name, dtype: object

In [46]:
## Tgain we are getting this error.. this could be due to data has some duplicate anime names.. also anime names seems not be in clean state..
## Lets fix this..
## First clean anime names..
df['name'] = df['name'].str.lower().str.strip()

In [47]:
## Lets not duplicates for now because there could be same show name with different type (movie, TV etc)..
## Lets recreate similarity matrix..
similarity_matrix = cosine_similarity(feature_matrix)

In [48]:
## rerun recommend function with slight modification which will make sure to avoid any error in case user input have errors related to upper/lower case and space..
def recommend_anime(anime_name, df, similarity_matrix):
    anime_name = anime_name.lower().strip()
    if anime_name not in df['name'].values: # check if anime exists
        return 'Anime not found!!'
    idx = df[df['name'] == anime_name].index[0] # get index
    similarity_scores = list(enumerate(similarity_matrix[idx])) # get similarity scores
    similarity_scores = sorted(similarity_scores, key = lambda x: x[1], reverse = True) # sorting the output
    similarity_scores = similarity_scores[1:11] # removing user input itself and taking out top 10 only.. 
    anime_indices = [i[0] for i in similarity_scores] # get indices
    #print(type(anime_indices))
    #print(anime_indices[:5])
    return df['name'].loc[anime_indices] # return names based on indices returned (anime_index)

In [49]:
## test function..
recommend_anime('Under World', df, similarity_matrix)

12203                          tenshi no habataki jun
12204                                the satisfaction
12176                         hokenshitsu de aimashou
12205    toushindai my lover: minami tai mecha-minami
12200                              super erotic anime
12183                                   lovely series
12185                              milky gal: cats ai
12163                                 prima donna mai
12197                                  sakura no mori
12192                           original c-v-p momoko
Name: name, dtype: object

In [50]:
## Recommendation with threshold condition.. 
## Right now function is returning top 10 anime only..now we want it to return only those animes whose similarty score > certain threshold value..

In [51]:
## For this we need to modify the function slightly.. add a threshold in it..
def recommend_anime_threshold(anime_name, df, similarity_matrix, threshold = 0.2): ## added threshold value
    anime_name = anime_name.lower().strip()
    
    if anime_name not in df['name'].values: 
        return 'Anime not found!!'
        
    idx = df[df['name'] == anime_name].index[0] 
    similarity_scores = list(enumerate(similarity_matrix[idx])) 
    similarity_scores = sorted(similarity_scores, key = lambda x: x[1], reverse = True) 

    threshold_scores = [i for i in similarity_scores if i[1] >= threshold] ## apply threshold..
    
    threshold_scores = threshold_scores[1:11]
    anime_indices = [i[0] for i in threshold_scores] 
    return df['name'].loc[anime_indices] 

In [52]:
recommend_anime_threshold('Natsume Yuujinchou Shi', df, similarity_matrix, threshold = 0.5)

46                                 natsume yuujinchou san
56                                zoku natsume yuujinchou
31                                  natsume yuujinchou go
155              natsume yuujinchou: itsuka yuki no hi ni
142                                    natsume yuujinchou
1040    natsume yuujinchou: nyanko-sensei to hajimete ...
10                                   clannad: after story
2144                                    fuujin monogatari
312     ano hi mita hana no namae wo bokutachi wa mada...
453                                      colorful (movie)
Name: name, dtype: object

In [53]:
## The output that we got is kind of a real world output.. 
## User input was 'gintama&#039;'
## If you look closely the output that we got seems to be totally related to 'gintama&#039;' only.. 
## All the other anime related to user input.. kind of remaining parts of the same movie franchise..

In [54]:
## lets run the same above function with minimising the threshold to 0.2
recommend_anime_threshold('Natsume Yuujinchou Shi', df, similarity_matrix, threshold = 0.2)

46                                 natsume yuujinchou san
56                                zoku natsume yuujinchou
31                                  natsume yuujinchou go
155              natsume yuujinchou: itsuka yuki no hi ni
142                                    natsume yuujinchou
1040    natsume yuujinchou: nyanko-sensei to hajimete ...
10                                   clannad: after story
2144                                    fuujin monogatari
312     ano hi mita hana no namae wo bokutachi wa mada...
453                                      colorful (movie)
Name: name, dtype: object

In [55]:
## With threshold = 0.9
recommend_anime_threshold('Natsume Yuujinchou Shi', df, similarity_matrix, threshold = 0.9)

46                                 natsume yuujinchou san
56                                zoku natsume yuujinchou
31                                  natsume yuujinchou go
155              natsume yuujinchou: itsuka yuki no hi ni
142                                    natsume yuujinchou
1040    natsume yuujinchou: nyanko-sensei to hajimete ...
Name: name, dtype: object

In [56]:
## I tried different anime to analyse the result and i found out that the anime 'Natsume Yuujinchou Shi' giving us the right output to interpret perfectly..
## It was observed that higher threshold values result in fewer recommendations, which are kind of subset of the top recommendations obtained at lower thresholds..
## This happened because the recommendations are sorted based on similarity and stricter thresholds filter out lower similarity items while preserving the ranking order..
## Higher threshold-->stricter results-->fewer items from top of the order..

## we can modify the recommendation system to more realistic by adding diversity (not all same type of anime)..
## And by adding popularity or something with respect to ratings and number of members..
## this will avoid getting too many same franchise results like we got..

## And sorting is important because it make sure that we get top recommendations with respect to similarity..
## If we skip sorting then recommendations will be in random order, not based on similarity strength..
## Threshold will still work but results will be totally unranked.. we might get least similar anime from the similarity list at the top..
## Without sorting the recommendation system will loose its ability to prioritize the most similar items..
## It will lead to poor and unreliable recommendations..

## Also slicing is must to limit the number of recommendation by selecting only the top X most similar items..
## It will also make the output concise and relevant.. 
## Nobody wants to go through 100s of recommendations.. so top 10 or top 20 can be chosen as slicing criteria according to the requirement..

### Evaluation:

In [57]:
## ● Split the dataset into training and testing sets.
## ● Evaluate the recommendation system using appropriate metrics such as precision, recall, and F1-score.
## ● Analyze the performance of the recommendation system and identify areas of improvement.

In [58]:
## Split the data..
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size = 0.2, random_state = 42)

In [59]:
## We will built similarity only on train dataset..
## Lets follow the same steps..

## Reset index..
train_df = train_df.reset_index(drop = True)
test_df = test_df.reset_index(drop = True )

In [60]:
## Feature engineering on trained data..
## Genre is already list so no split has to be done
mlb = MultiLabelBinarizer()
genre_encoded_train = mlb.fit_transform(train_df['genre'])
genre_df_train = pd.DataFrame(genre_encoded_train, columns = mlb.classes_)

In [61]:
## Scalling numerical features..using minmax scaler as above..
scaler = MinMaxScaler()
num_features_train = scaler.fit_transform(train_df[['rating', 'episodes', 'members']])
num_df_train = pd.DataFrame(num_features_train, columns = ['rating', 'episodes', 'members'])

In [62]:
## Now lets combine both..
train_features = pd.concat([genre_df_train, num_df_train], axis = 1)

In [63]:
## Now apply cosine similarity..
similarity_matrix_train = cosine_similarity(train_features)

In [64]:
## Now lets create evaluation function..
def evaluate_model(train_df, test_df, similarity_matrix_train, top_n = 10):
    precisions = []
    recalls = []

    for i in range(len(test_df)):
        test_genres = set(test_df.iloc[i]['genre'])
        idx = i % len(train_df)

        similarity_scores = list(enumerate(similarity_matrix_train[idx]))
        similarity_scores = sorted(similarity_scores, key = lambda x: x[1], reverse = True)[1:top_n+1]

        indices = [x[0] for x in similarity_scores]

        recommended = train_df.iloc[indices]
        
        rec_genres = set()
        for g in recommended['genre']:
            rec_genres.update(g)

        intersection = test_genres.intersection(rec_genres)

        precision = len(intersection) / len(rec_genres) if len(rec_genres) > 0 else 0
        recall = len(intersection) / len(test_genres) if len(test_genres) > 0 else 0

        precisions.append(precision)
        recalls.append(recall)

    avg_precision = sum(precisions) /len(precisions)
    avg_recall = sum(recalls) / len(recalls)

    f1 = (2 * avg_precision *avg_recall) / (avg_precision + avg_recall) if (avg_precision + avg_recall) > 0 else 0
    return avg_precision, avg_recall, f1

In [65]:
precision , recall, f1 = evaluate_model(train_df, test_df, similarity_matrix_train)

print('Precision:', precision)
print('Recall:', recall)
print('F1 Score:', f1)

Precision: 0.10135822232114099
Recall: 0.1321801571801572
F1 Score: 0.11473528065510327


In [66]:
## The precision is relatively low (10%) which indicates that only small portion of the recommended  anime share common genres with the test anime..
## Thew recall is slightly higher (13%) suggesting that the model is able to capture some relevant genres from the test anime.. but still misses many..
## The F1 score (0.11 reflects overall moderate performance, showing that there is a trade off between precision and recall..

## This means recommendation system is able to find some similar animes, but not highly accurate in capturing all relevant features..
## The reason for these low scores would be absence of important features like user preference, watch history, personalised behaviors etc..


### Interview Questions:

In [67]:
## ● 1. Can you explain the difference between user-based and item-based collaborative filtering?
## ● 2. What is collaborative filtering, and how does it work?

In [68]:
## User-based Collaborative filtering recommends items by finding the user who have similar interests or behaviour..
## If two users  like similar movies, products or songs, the system recommends items liked by one user to the other user..
## On the other hand item-based collaborative filtering recommends items based on similarities between items.. 
## If a user likes a particular product or movie, the system suggests similar products or movies.. 
## User-based filtering focuses on similar users, while item based filtering focuses on similar items..
## Item-based collaborative filtering is more commonly used because it works faster and is more scalable for large dataset.. 

In [69]:
## Collaborative filtering is a recommendation technique usedto suggest products,movies, or songs or other items to the users based on their interest and behaviour..
## It works by collecting user data such as ratings, purchases, clicksor watch history and findings similarities between users or items..
## If user with similar preference liked a particular item, the system recommends that item to other similar users..
## Collaborative filtering mailyworks in two ways: User-based filtering and item-based filtering..
## This technique is widely used in platforms like netflix, amazon and spotify to provide personalised recommendations..